In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.integrate import quad

cwd = Path.cwd()
if (cwd / 'heavy_tailed_mlp.md').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'random_dnn' / 'heavy_tailed_mlp.md').exists():
    sys.path.insert(0, str(cwd / 'random_dnn'))

rng_global = np.random.default_rng(0)

_timings = []  # (label, seconds) running log of cell-level timings

class Timer:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *a):
        dt = time.perf_counter() - self.t0
        _timings.append((self.label, dt))
        print(f'[{self.label}] {dt:.2f} s')

# Heavy-tailed MLP: empirical validation of the perturbation recursion

Companion notebook to `heavy_tailed_mlp.md`. The derivation conjectures a
linear recursion for the perturbation $\alpha$-scale at the fully-correlated
fixed point,

$$
(T^l)^\alpha = \chi_1|_\alpha\, (T^{l-1})^\alpha,
\qquad
\chi_1|_\alpha = \sigma_w^\alpha \int p_\alpha(z)\,|\phi'((q^*)^{1/\alpha} z)|^\alpha\,dz,
$$

which rests on a **mean-field decoupling assumption** -- that the joint
$(\phi'(h^{l-1}_j), \xi^{l-1}_j)$ at a single neuron factorises in the
infinite-width limit. At $\alpha = 2$ the decoupling is rigorous (Gaussian
covariance argument). For $\alpha < 2$ it is conjectural and the
linearisation that derives it from the exact recursion is *non-uniform*:
$\mathbb E|\phi'(h)\xi|^\alpha$ formally diverges because $|\xi|^\alpha$ has
infinite mean for an exactly $\alpha$-stable $\xi$. The notebook compares
the linear-recursion prediction against the *exact* recursion

$$
(T^l)^\alpha = \sigma_w^\alpha\, \mathbb E\big|\phi(h+\xi)-\phi(h)\big|^\alpha
$$

(which is unconditional, finite for bounded $\phi$, and is what eq. (10) of
the .md file actually says) and against direct simulation.

Three tests, in order of cost and strength:

- **Test 1**: empirical moment-factorisation ratio $R_N^l \to 1$ as $N \to \infty$?
  If yes, the weak decoupling holds.
- **Test 2**: measured $T^l$ vs (a) linear-recursion prediction
  $\chi_1|_\alpha^{l/\alpha} T^0$, (b) exact-recursion prediction iterated
  from $T^0$. If (a) and (b) agree, the linear recursion is valid. If they
  disagree, the linearisation is non-uniform and the simple $\chi_1|_\alpha$
  picture is wrong.
- **Test 3**: empirical joint spectral measure of $(h, \xi)$. If it
  concentrates on the coordinate axes, the strong-form decoupling holds.

Convention (matches the .md file): symmetric $\alpha$-stable with
characteristic function $\hat p_\alpha(t) = \exp(-\tfrac{1}{2}|t|^\alpha)$,
weights $W^l_{ij} = \sigma_w\,N^{-1/\alpha}\,\tilde W^l_{ij}$ with
$\tilde W \sim p_\alpha$. At $\alpha = 2$, $\tilde W \sim \mathcal N(0,1)$
and Poole's conventions are recovered exactly.

In [ ]:
DEFAULT_PARAMS = dict(
    alpha=1.5,
    sigma_w=1.0,
    depth=12,
    widths_t1=[64, 256, 1024, 4096],   # Test 1 sweep
    width_t2=2048,                      # Test 2 fixed width
    width_t3=2048,                      # Test 3 fixed width
    num_reps_t1=8,
    num_reps_t2=16,
    num_reps_t3=32,
    T0=1e-3,                            # Test 2 initial perturbation scale
    n_quad=400,                         # density grid points for theoretical integrals
)
DEFAULT_PARAMS

In [ ]:
# ----- Stable sampling (Chambers-Mallows-Stuck), .md convention -----
# .md convention: tilde W has CF exp(-|t|^a / 2).
# Scipy convention: scale s gives CF exp(-|s t|^a).
# Relation: tilde W = scipy_stable(alpha, beta=0, scale=2**(-1/alpha)).
# Equivalently, sample scipy-standard z and multiply by 2**(-1/alpha).

def sample_stable_md(alpha, size, rng):
    """Sample i.i.d. tilde W with .md-convention CF exp(-|t|^a / 2)."""
    if alpha == 2.0:
        return rng.standard_normal(size)
    u = rng.uniform(-np.pi/2, np.pi/2, size=size)
    w = rng.exponential(1.0, size=size)
    z = (np.sin(alpha * u) / np.cos(u) ** (1.0/alpha)
         * (np.cos((1.0 - alpha) * u) / w) ** ((1.0 - alpha)/alpha))
    return z * 2.0 ** (-1.0/alpha)  # convert scipy -> md

def pdf_stable_md(z, alpha):
    """.md-convention symmetric stable density p_alpha(z)."""
    return stats.levy_stable.pdf(z, alpha, beta=0, loc=0, scale=2.0**(-1.0/alpha))

# Sanity: at alpha = 2, p_2(z) should equal Gaussian density.
z_test = np.linspace(-3, 3, 7)
max_err = float(np.max(np.abs(pdf_stable_md(z_test, 2.0) - stats.norm.pdf(z_test))))
print(f'alpha=2 density vs standard normal: max abs err = {max_err:.2e}')

In [ ]:
# ----- Theoretical fixed point q^* and slope chi_1|_alpha -----
# Optimisation: cache the .md-convention PDF on a fixed grid per alpha.
# scipy.stats.levy_stable.pdf is slow (~10-50 ms per scalar eval for alpha<2),
# but vectorises cheaply. Compute once per alpha, reuse for V_alpha/chi.

_pdf_cache = {}

def _stable_grid(alpha, z_max=80.0, n_grid=4001):
    """Cached (z, p_alpha(z)) on a uniform grid; PDF computed once per alpha."""
    key = (float(alpha), z_max, n_grid)
    if key not in _pdf_cache:
        t0 = time.perf_counter()
        z = np.linspace(-z_max, z_max, n_grid)
        pdf = pdf_stable_md(z, alpha)
        dt = time.perf_counter() - t0
        _pdf_cache[key] = (z, pdf)
        _timings.append((f'_stable_grid(alpha={alpha})', dt))
        print(f'[_stable_grid] alpha={alpha}: built {n_grid}-point PDF grid in {dt:.2f} s')
    return _pdf_cache[key]

def V_alpha(q, alpha, sigma_w, phi):
    """Length-map V_alpha(q) via cached grid + trapezoid."""
    if q <= 0:
        return 0.0
    s = q ** (1.0/alpha)
    z, pdf = _stable_grid(alpha)
    return sigma_w**alpha * float(np.trapz(pdf * np.abs(phi(s * z))**alpha, z))

def length_fixed_point(alpha, sigma_w, phi, max_iter=200, tol=1e-9):
    """Iterate V_alpha to find q^*. For alpha<2 with bounded phi, q^*>0 always exists."""
    q = 1.0
    for _ in range(max_iter):
        q_new = V_alpha(q, alpha, sigma_w, phi)
        if abs(q_new - q) < tol * max(q, 1e-12):
            return q_new
        q = q_new
    return q

def chi_1_alpha(alpha, sigma_w, q_star, phi_prime):
    s = q_star ** (1.0/alpha)
    z, pdf = _stable_grid(alpha)
    return sigma_w**alpha * float(np.trapz(pdf * np.abs(phi_prime(s * z))**alpha, z))

def exact_recursion_T_alpha(T_prev, alpha, sigma_w, q_star, phi, n_mc=8000, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    h = q_star ** (1.0/alpha) * sample_stable_md(alpha, n_mc, rng)
    xi = T_prev * sample_stable_md(alpha, n_mc, rng)
    return sigma_w ** alpha * float(np.mean(np.abs(phi(h + xi) - phi(h)) ** alpha))

phi = np.tanh
phi_p = lambda h: 1.0 - np.tanh(h)**2
with Timer('cell4 sanity (alpha=2 sweep, three sigma_w)'):
    for sw in [1.5, 2.0, 3.0]:
        qs = length_fixed_point(2.0, sw, phi)
        chi = chi_1_alpha(2.0, sw, qs, phi_p)
        print(f'  alpha=2, sigma_w={sw}: q* = {qs:.4f}, chi_1 = {chi:.4f}')

In [ ]:
# ----- Forward pass with two nearby inputs -----

def forward_two_inputs(alpha, sigma_w, N, L, T0, phi, phi_prime, rng, q_star=None):
    """Single MLP realisation forward with two inputs differing by a small perturbation.

    Returns dict of length-L arrays: h1, h2, xi, phi_prime_h1, and per-layer R.
    Pass q_star to avoid recomputing the length fixed point per realisation.
    """
    if q_star is None:
        q_star = length_fixed_point(alpha, sigma_w, phi)
    h1 = q_star ** (1.0/alpha) * sample_stable_md(alpha, N, rng)
    xi0 = T0 * sample_stable_md(alpha, N, rng)
    h2 = h1 + xi0
    out = {'h1': [], 'h2': [], 'xi': [], 'phip_h1': [], 'R': []}
    for l in range(L):
        a1 = phi(h1); a2 = phi(h2)
        W = sigma_w * N ** (-1.0/alpha) * sample_stable_md(alpha, (N, N), rng)
        h1, h2 = W @ a1, W @ a2
        xi = h2 - h1
        phip = phi_prime(h1)
        A = float(np.mean(np.abs(phip * xi) ** alpha))
        B = float(np.mean(np.abs(phip) ** alpha)) * float(np.mean(np.abs(xi) ** alpha))
        R = A / B if B > 0 else np.nan
        out['h1'].append(h1.copy()); out['h2'].append(h2.copy())
        out['xi'].append(xi.copy()); out['phip_h1'].append(phip.copy())
        out['R'].append(R)
    for k in out:
        out[k] = np.array(out[k])
    return out

# Quick validation (one forward pass at N=512)
with Timer('cell5 forward N=512 L=8'):
    q_star_test = length_fixed_point(1.5, 2.0, phi)
    out = forward_two_inputs(
        alpha=1.5, sigma_w=2.0, N=512, L=8,
        T0=1e-3, phi=phi, phi_prime=phi_p, rng=rng_global, q_star=q_star_test,
    )
print(f'alpha=1.5 sigma_w=2.0: q* = {q_star_test:.4f}')
print('Per-layer R:', [f'{r:.3f}' for r in out['R']])

## Test 0a: Length-map convergence

Sanity check of the iterated length map $V_\alpha$ against single-input forward simulation. At each layer estimate the empirical $\alpha$-scale $\hat S^l$ of $h^l$ from the characteristic-function estimator and compare $\hat q^l := (\hat S^l)^\alpha$ to the theoretical iteration $q^l = V_\alpha(q^{l-1})$ from a chosen initial $q^0$. Convergence to $q^*(\alpha, \sigma_w)$ within $\sim 5$ layers is expected for $\alpha < 2$.

In [ ]:
def empirical_S_alpha(h, alpha):
    # Empirical alpha-scale of h via CF estimator (md convention CF = exp(-|t|^alpha/2)).
    iqr = float(np.percentile(np.abs(h), 75))
    t = 1.0 / max(iqr, 1e-12)
    phi_hat = max(float(np.mean(np.cos(t * h))), 1e-10)
    return (-2.0 * np.log(phi_hat) / (abs(t) ** alpha)) ** (1.0/alpha)

phi = np.tanh
phi_p = lambda h: 1.0 - np.tanh(h)**2

test0_alphas = [1.3, 1.5, 1.8, 2.0]
test0_sigma_w = 1.5
test0_N = 2048
test0_L = 15
test0_q0_set = [0.05, 1.0, 10.0]
rng_t0 = np.random.default_rng(0)

with Timer(f'Test0a forward sweep: {len(test0_alphas)} alphas x {len(test0_q0_set)} q0 x N={test0_N}'):
    fig, axes = plt.subplots(1, len(test0_alphas), figsize=(4*len(test0_alphas), 4), sharey=True)
    for ax, alpha in zip(axes, test0_alphas):
        q_star = length_fixed_point(alpha, test0_sigma_w, phi)
        for q0 in test0_q0_set:
            out = forward_two_inputs(alpha=alpha, sigma_w=test0_sigma_w, N=test0_N, L=test0_L,
                                     T0=1e-6, phi=phi, phi_prime=phi_p, rng=rng_t0, q_star=q0)
            q_emp = np.array([empirical_S_alpha(out['h1'][l], alpha)**alpha for l in range(test0_L)])
            q_th = np.empty(test0_L); q = q0
            for l in range(test0_L):
                q = V_alpha(q, alpha, test0_sigma_w, phi); q_th[l] = q
            line, = ax.semilogy(range(1, test0_L+1), q_emp, 'o-', label=f'emp $q^0={q0}$', alpha=0.8, markersize=5)
            ax.semilogy(range(1, test0_L+1), q_th, '--', color=line.get_color(), alpha=0.9)
        ax.axhline(q_star, color='k', linestyle=':', alpha=0.6, label=f'$q^*={q_star:.3f}$')
        ax.set_title(f'$\\alpha={alpha}$, $\\sigma_w={test0_sigma_w}$')
        ax.set_xlabel('layer $l$')
        if ax is axes[0]: ax.set_ylabel('$q^l$')
        ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## Test 0b: $V_\alpha$ shape near $q=0$ and $q^*$ heatmap

Sec 4(d) of the derivation predicts $V_\alpha(q) \sim C\,q\log(1/q)$ as $q \to 0^+$ for $\alpha < 2$, giving $V_\alpha'(0^+) = +\infty$ (vertical tangent) and a unique $q^* > 0$ for every $\sigma_w > 0$. Two checks:

1. Plot $V_\alpha(q)/q$ on a log-$q$ axis; for $\alpha<2$ it should diverge like $\log(1/q)$, while for $\alpha=2$ it should be bounded.
2. Heatmap $q^*$ over a $(\alpha, \sigma_w)$ grid; for $\alpha<2$ no grid point should hit $q^* = 0$.

In [ ]:
# (1) Shape of V_alpha(q)/q near q=0
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
qs = np.logspace(-10, 1, 60)
with Timer(f'Test0b V_alpha/q shape ({len(qs)} q points x 5 alphas)'):
    for alpha in [1.3, 1.5, 1.8, 1.95, 2.0]:
        v = np.array([V_alpha(q, alpha, 1.5, phi) for q in qs])
        ax1.loglog(qs, v / qs, '-', label=f'$\\alpha={alpha}$')
qref = qs[qs < 1e-2]
ax1.loglog(qref, 0.15*np.log(1/qref), 'k:', alpha=0.5, label='$\\propto\\log(1/q)$ ref')
ax1.set_xlabel('$q$'); ax1.set_ylabel('$V_\\alpha(q)/q$')
ax1.set_title('Slope blow-up at $q=0$ for $\\alpha<2$')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3, which='both')

# (2) q* heatmap over (alpha, sigma_w)
alpha_grid = np.linspace(1.1, 2.0, 10)
sigma_grid = np.linspace(0.2, 3.0, 15)
with Timer(f'Test0b q* heatmap ({len(alpha_grid)}x{len(sigma_grid)})'):
    Qstar = np.empty((len(alpha_grid), len(sigma_grid)))
    for i, a in enumerate(alpha_grid):
        for j, sw in enumerate(sigma_grid):
            Qstar[i, j] = length_fixed_point(a, sw, phi)
n_zero = int(np.sum(Qstar <= 1e-8))
im = ax2.pcolormesh(sigma_grid, alpha_grid, np.log10(np.maximum(Qstar, 1e-10)), shading='auto')
plt.colorbar(im, ax=ax2, label='$\\log_{10} q^*$')
ax2.set_xlabel('$\\sigma_w$'); ax2.set_ylabel('$\\alpha$')
ax2.set_title(f'$q^*(\\alpha,\\sigma_w)$ -- {n_zero}/{Qstar.size} points with $q^*\\leq 10^{{-8}}$')
plt.tight_layout(); plt.show()
print(f'Test 0b: q*>1e-8 at {Qstar.size - n_zero}/{Qstar.size} grid points')
print(f'  min q* over alpha<2: {Qstar[alpha_grid<2].min():.3g}')
print(f'  min q* at alpha=2.0: {Qstar[alpha_grid==2.0].min():.3g}  (Gaussian: 0 below threshold)')

## Test 1: Moment-factorisation ratio $R_N^l \to 1$?

If the weak-form decoupling holds, $R_N^l = (1/N) \sum_j |\phi'(h_j) \xi_j|^\alpha / [\overline{|\phi'|^\alpha} \cdot \overline{|\xi|^\alpha}]$ should approach 1 as $N \to \infty$, at every layer $l$. We sweep $N$ and plot $|R - 1|$ on log-log axes.

In [ ]:
p = DEFAULT_PARAMS
rng_t1 = np.random.default_rng(1)
with Timer('Test1 fixed-point setup'):
    q_star_t1 = length_fixed_point(p['alpha'], p['sigma_w'], phi)
print(f"Test 1: alpha={p['alpha']}, sigma_w={p['sigma_w']}, q* = {q_star_t1:.4f}")

R_data = {}
with Timer('Test1 total forward sweep'):
    for N in p['widths_t1']:
        with Timer(f'Test1 N={N} ({p["num_reps_t1"]} reps x {p["depth"]} layers)'):
            R_layers = np.zeros((p['num_reps_t1'], p['depth']))
            for r in range(p['num_reps_t1']):
                out = forward_two_inputs(
                    alpha=p['alpha'], sigma_w=p['sigma_w'], N=N, L=p['depth'],
                    T0=p['T0'], phi=phi, phi_prime=phi_p, rng=rng_t1, q_star=q_star_t1,
                )
                R_layers[r] = out['R']
            R_data[N] = R_layers
        print(f'  N={N}: median R per layer = {np.median(R_layers, axis=0).round(3).tolist()}')

with Timer('Test1 plotting'):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]
    mid_layer = p['depth'] // 2
    Ns = np.array(p['widths_t1'])
    med = np.array([np.median(np.abs(R_data[N][:, mid_layer] - 1)) for N in Ns])
    ax.loglog(Ns, med, 'o-', label=f'median |R-1| at layer {mid_layer}')
    ax.loglog(Ns, 1.0 / Ns**0.5, ':', alpha=0.4, label='$N^{-1/2}$ guide')
    ax.set_xlabel('width $N$'); ax.set_ylabel('$|R_N^l - 1|$')
    ax.set_title(f'Test 1: weak decoupling, $\\alpha={p["alpha"]}$, layer ${mid_layer}$')
    ax.legend(); ax.grid(alpha=0.3)
    ax = axes[1]
    for N in p['widths_t1']:
        ax.plot(np.median(R_data[N], axis=0), 'o-', label=f'N={N}', alpha=0.7)
    ax.axhline(1.0, color='k', ls='--', alpha=0.5, label='decoupling target')
    ax.set_xlabel('layer $l$'); ax.set_ylabel('median $R_N^l$')
    ax.set_title('Test 1: layerwise')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## Test 2: predicted vs measured perturbation growth

Three predictions to compare against simulation:
1. **Linear recursion** (conditional on decoupling): $(T^l)^\alpha = \chi_1|_\alpha (T^{l-1})^\alpha$.
2. **Exact recursion**: $(T^l)^\alpha = \sigma_w^\alpha \mathbb E|\phi(h+\xi)-\phi(h)|^\alpha$ with $(h,\xi)$ independent stables of the right scales -- this is eq. (10) of the .md, iterated by feeding $T^l$ back as $T^{l-1}$.
3. **Empirical**: $\hat T^l$ extracted from the empirical CF $\widehat{\varphi}_\xi(t)$ via $\hat T^\alpha = -2 \log|\hat \varphi(t)| / |t|^\alpha$.

If (1) and (2) agree across depth, the linear recursion is a good description and $\chi_1|_\alpha$ is the operational slope. If they diverge, the linearisation is non-uniform and we should believe (2) over (1).

In [ ]:
def empirical_T_alpha(xi, alpha, t=None):
    if t is None:
        iqr = float(np.percentile(np.abs(xi), 75))
        t = 1.0 / max(iqr, 1e-12)
    phi_hat = float(np.mean(np.cos(t * xi)))
    phi_hat = max(phi_hat, 1e-10)
    return -2.0 * np.log(phi_hat) / (abs(t) ** alpha)

p = DEFAULT_PARAMS
rng_t2 = np.random.default_rng(2)
alpha = p['alpha']; sigma_w = p['sigma_w']
with Timer('Test2 fixed-point + chi'):
    q_star = length_fixed_point(alpha, sigma_w, phi)
    chi = chi_1_alpha(alpha, sigma_w, q_star, phi_p)
print(f'Test 2: alpha={alpha}, sigma_w={sigma_w}, q* = {q_star:.4f}, chi_1|_alpha = {chi:.4f}')

T_emp = np.zeros((p['num_reps_t2'], p['depth']))
with Timer(f'Test2 forward sweep ({p["num_reps_t2"]} reps x N={p["width_t2"]} x L={p["depth"]})'):
    for r in range(p['num_reps_t2']):
        out = forward_two_inputs(
            alpha=alpha, sigma_w=sigma_w, N=p['width_t2'], L=p['depth'],
            T0=p['T0'], phi=phi, phi_prime=phi_p, rng=rng_t2, q_star=q_star,
        )
        for l in range(p['depth']):
            T_emp[r, l] = empirical_T_alpha(out['xi'][l], alpha)
T_emp_med = np.median(T_emp, axis=0)

T_lin = (p['T0'] ** alpha) * chi ** np.arange(1, p['depth'] + 1)
T_exact = np.zeros(p['depth'])
T_prev_alpha = p['T0'] ** alpha
rng_mc = np.random.default_rng(100)
with Timer(f'Test2 exact recursion ({p["depth"]} steps, n_mc=8000)'):
    for l in range(p['depth']):
        T_prev_alpha = exact_recursion_T_alpha(
            T_prev_alpha ** (1.0/alpha), alpha, sigma_w, q_star, phi, n_mc=8000, rng=rng_mc,
        )
        T_exact[l] = T_prev_alpha

with Timer('Test2 plotting'):
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.semilogy(range(1, p['depth']+1), T_emp_med, 'o-', label='empirical $\\hat T^l$', markersize=6)
    ax.semilogy(range(1, p['depth']+1), T_lin ** (1.0/alpha), 's--', label='linear: $\\chi_1|_\\alpha^{l/\\alpha} T^0$')
    ax.semilogy(range(1, p['depth']+1), T_exact ** (1.0/alpha), '^:', label='exact iterated recursion (10)')
    ax.set_xlabel('layer $l$'); ax.set_ylabel('$T^l$ (perturbation scale)')
    ax.set_title(f'Test 2: perturbation growth, $\\alpha={alpha}$, $\\sigma_w={sigma_w}$, $N={p["width_t2"]}$')
    ax.legend(); ax.grid(alpha=0.3, which='both')
    plt.tight_layout(); plt.show()

## Test 3: empirical joint spectral measure of $(h, \xi)$

Strong form of decoupling: the joint $(h^{l-1}_j, \xi^{l-1}_j)$ is bivariate symmetric $\alpha$-stable, and decoupling requires its spectral measure $\Gamma_{(h,\xi)}$ to concentrate on $\pm e_h, \pm e_\xi$ (coordinate axes). We estimate $\Gamma$ from samples by collecting many $(h, \xi)$ pairs across neurons and realisations, then plotting the angular distribution of points with large radius (where the spectral measure dominates), weighted by $R^\alpha$.

In [ ]:
p = DEFAULT_PARAMS
alpha = p['alpha']; sigma_w = p['sigma_w']
rng_t3 = np.random.default_rng(3)
with Timer('Test3 fixed-point'):
    q_star = length_fixed_point(alpha, sigma_w, phi)

target_layer = 4
N_t3 = 1024
reps_t3 = 16
h_samples, xi_samples = [], []
with Timer(f'Test3 forward sweep ({reps_t3} reps x N={N_t3} x L={target_layer+1})'):
    for r in range(reps_t3):
        out = forward_two_inputs(
            alpha=alpha, sigma_w=sigma_w, N=N_t3, L=target_layer + 1,
            T0=p['T0'], phi=phi, phi_prime=phi_p, rng=rng_t3, q_star=q_star,
        )
        h_samples.append(out['h1'][target_layer])
        xi_samples.append(out['xi'][target_layer])
h = np.concatenate(h_samples); xi = np.concatenate(xi_samples)
T_h_alpha = empirical_T_alpha(h, alpha)
T_xi_alpha = empirical_T_alpha(xi, alpha)
h_n = h / max(T_h_alpha ** (1.0/alpha), 1e-12)
xi_n = xi / max(T_xi_alpha ** (1.0/alpha), 1e-12)
R = np.sqrt(h_n**2 + xi_n**2)
Theta = np.arctan2(xi_n, h_n)

with Timer('Test3 plotting'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    idx = np.random.default_rng(0).choice(len(R), size=min(5000, len(R)), replace=False)
    ax.scatter(h_n[idx], xi_n[idx], s=2, alpha=0.3)
    ax.set_xlim(-30, 30); ax.set_ylim(-30, 30)
    ax.axhline(0, color='r', ls='--', alpha=0.4)
    ax.axvline(0, color='r', ls='--', alpha=0.4)
    ax.set_xlabel('$h$ (normalised)'); ax.set_ylabel('$\\xi$ (normalised)')
    ax.set_title(f'Test 3: joint scatter at layer {target_layer}')
    ax.set_aspect('equal')
    ax = axes[1]
    weights = R ** alpha
    hist, edges = np.histogram(Theta, bins=72, range=(-np.pi, np.pi), weights=weights, density=False)
    centers = 0.5 * (edges[1:] + edges[:-1])
    ax.plot(centers, hist / hist.max(), 'b-')
    for theta0 in [-np.pi, -np.pi/2, 0, np.pi/2]:
        ax.axvline(theta0, color='r', ls=':', alpha=0.4)
    ax.set_xlabel('$\\theta$ (rad)'); ax.set_ylabel('angular density (normalised)')
    ax.set_title('Test 3: angular distribution, $R^\\alpha$-weighted')
    ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_xticklabels(['$-\\pi$', '$-\\pi/2$', '$0$', '$\\pi/2$', '$\\pi$'])
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

axis_band = np.pi / 8
near_axes_mask = (
    (np.abs(Theta) < axis_band)
    | (np.abs(np.abs(Theta) - np.pi) < axis_band)
    | (np.abs(np.abs(Theta) - np.pi/2) < axis_band)
)
frac_near_axes = float(np.sum(weights[near_axes_mask]) / np.sum(weights))
frac_uniform = (4 * 2 * axis_band) / (2 * np.pi)
print(f'fraction of R^alpha mass within +/-{axis_band:.2f} rad of an axis: {frac_near_axes:.3f}')
print(f'(uniform angular distribution would give: {frac_uniform:.3f})')

print()
print('=== TIMING SUMMARY (all cells) ===')
for label, dt in _timings:
    print(f'  {dt:8.2f} s   {label}')
print(f'  {sum(d for _, d in _timings):8.2f} s   TOTAL')

## Test 4: Empirical edge of chaos vs naive $\chi_1|_\alpha^{\text{naive}} = 1$ over $(\alpha, \sigma_w)$

Per-grid-point: 3-rep forward MLP at width $N=1024$, depth $L=25$, initial perturbation $T^0 = 10^{-3}$. Median across reps stabilises noise. Slope fitted to $\log_{10} T^l$ over layers $\ge L/2$ (avoid initial transient). Grid points with truncated $q^* < 10^{-4}$ are masked (the trivial-fixed-point linear regime, where naive analysis agrees with the empirical for mechanical reasons unrelated to heavy-tail structure -- see Section 4(d) of `heavy_tailed_mlp.md`).

If the black (empirical EoC) and magenta (naive) contours coincide, the naive scalar slope captures perturbation dynamics. They are expected to disagree based on Tests 1--3.

In [ ]:
t4_alpha_grid = np.linspace(1.1, 2.0, 10)
t4_sigma_grid = np.linspace(0.5, 2.5, 11)
t4_N = 1024
t4_L = 25
t4_T0 = 1e-3
t4_reps = 3
t4_q_threshold = 1e-4
rng_t4 = np.random.default_rng(7)

growth_rate = np.full((len(t4_alpha_grid), len(t4_sigma_grid)), np.nan)
chi_naive = np.full_like(growth_rate, np.nan)
qstar_grid = np.full_like(growth_rate, np.nan)

with Timer(f'Test4 critical-line scan ({len(t4_alpha_grid)}x{len(t4_sigma_grid)} reps={t4_reps} N={t4_N} L={t4_L})'):
    for i, a in enumerate(t4_alpha_grid):
        for j, sw in enumerate(t4_sigma_grid):
            q_star = length_fixed_point(a, sw, phi)
            qstar_grid[i, j] = q_star
            if q_star < t4_q_threshold:
                continue
            chi_naive[i, j] = chi_1_alpha(a, sw, q_star, phi_p)
            T_l_reps = []
            for r in range(t4_reps):
                out = forward_two_inputs(alpha=a, sigma_w=sw, N=t4_N, L=t4_L, T0=t4_T0,
                                         phi=phi, phi_prime=phi_p, rng=rng_t4, q_star=q_star)
                T_l_reps.append([empirical_T_alpha(out['xi'][l], a) for l in range(t4_L)])
            T_l_med = np.median(np.maximum(T_l_reps, 1e-30), axis=0)
            T_l_S = T_l_med ** (1.0 / a)
            ll = np.arange(t4_L // 2, t4_L)
            slope = np.polyfit(ll, np.log10(T_l_S[ll]), 1)[0]
            growth_rate[i, j] = slope

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mask_nan = np.isnan(growth_rate)

ax = axes[0]
vmax = max(0.05, float(np.nanmax(np.abs(growth_rate))))
im = ax.pcolormesh(t4_sigma_grid, t4_alpha_grid, growth_rate, shading='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(im, ax=ax, label='per-layer $\\log_{10}$ growth of $T^l$')
masked_growth = np.where(mask_nan, np.nan, growth_rate)
masked_chi = np.where(mask_nan, np.nan, chi_naive - 1.0)
ax.contour(t4_sigma_grid, t4_alpha_grid, masked_growth, levels=[0], colors='k', linewidths=2)
ax.contour(t4_sigma_grid, t4_alpha_grid, masked_chi, levels=[0], colors='magenta', linewidths=2, linestyles='--')
ax.contourf(t4_sigma_grid, t4_alpha_grid, mask_nan.astype(float), levels=[0.5, 1.5], hatches=['xxx'], colors='none', alpha=0.5)
ax.set_xlabel('$\\sigma_w$'); ax.set_ylabel('$\\alpha$')
ax.set_title('Empirical EoC (black) vs naive $\\chi_1|_\\alpha=1$ (magenta dashed); hatched: $q^*<10^{-4}$')

ax = axes[1]
chi_disp = np.where(mask_nan, np.nan, chi_naive)
im2 = ax.pcolormesh(t4_sigma_grid, t4_alpha_grid, chi_disp, shading='auto', cmap='viridis')
plt.colorbar(im2, ax=ax, label='$\\chi_1|_\\alpha^{\\mathrm{naive}}$')
ax.contour(t4_sigma_grid, t4_alpha_grid, masked_chi, levels=[0], colors='magenta', linewidths=2, linestyles='--')
ax.set_xlabel('$\\sigma_w$'); ax.set_ylabel('$\\alpha$')
ax.set_title('Naive $\\chi_1|_\\alpha^{\\mathrm{naive}}$ evaluated at $q^*$')

plt.tight_layout(); plt.show()

n_masked = int(mask_nan.sum())
n_valid = int((~mask_nan).sum())
print(f'Test 4: {n_valid}/{growth_rate.size} valid grid points, {n_masked} masked (q*<1e-4)')
print(f'  growth_rate range over valid grid: [{np.nanmin(growth_rate):.3f}, {np.nanmax(growth_rate):.3f}]')
print(f'  chi_naive   range over valid grid: [{np.nanmin(chi_naive):.3f}, {np.nanmax(chi_naive):.3f}]')
print(f'  approximate alpha-stripe correlations between empirical-EoC and naive=1:')
for i, a in enumerate(t4_alpha_grid):
    valid_j = np.where(~mask_nan[i])[0]
    if len(valid_j) < 3: continue
    # for each alpha row, find sigma_w where growth crosses 0 and where chi crosses 1
    g_row = growth_rate[i, valid_j]; c_row = chi_naive[i, valid_j] - 1.0
    sw_row = t4_sigma_grid[valid_j]
    try:
        sw_emp = float(np.interp(0.0, g_row, sw_row)) if g_row.min() < 0 < g_row.max() else float('nan')
        sw_naive = float(np.interp(0.0, c_row, sw_row)) if c_row.min() < 0 < c_row.max() else float('nan')
        print(f'    alpha={a:.2f}: empirical sw_c={sw_emp:.3f}, naive sw_c={sw_naive:.3f}, diff={sw_emp-sw_naive:+.3f}')
    except Exception as e:
        print(f'    alpha={a:.2f}: interp failed ({e})')

## Test 5: Direct verification of the bivariate-stable closure (eq 8)

Sections 5 and 8 of `heavy_tailed_mlp.md` rely on the bivariate-stable CF closure (eq 8): the layer-$l$ joint $(h^l_i(x^1), h^l_i(x^2))$ in the infinite-width limit is bivariate symmetric $\alpha$-stable with characteristic function

$$
\hat p^l(t_1, t_2) = \exp\!\left(-\tfrac{1}{2}\,\sigma_w^\alpha\, \mathbb{E}_{(h^{l-1}_1, h^{l-1}_2)}\, |t_1\, \phi(h^{l-1}_1) + t_2\, \phi(h^{l-1}_2)|^\alpha\right).
$$

Direct empirical check: at finite $N$ run the forward pass to layer $l$, then compare
- *empirical* $\log|\hat{\varphi}^l(t_1, t_2)|$ from the $N$-neuron sample, with
- *predicted* $\log|\hat p^l(t_1, t_2)| = -\tfrac{1}{2}\sigma_w^\alpha\, (1/N)\sum_j |t_1 \phi(h^{l-1}_j(x^1)) + t_2 \phi(h^{l-1}_j(x^2))|^\alpha$ from the previous-layer activations.

Probed on a fan of directions $\theta_t \in [0, \pi]$ at magnitudes $|t| \in [0.1, 2.0]$. RMSE and max-abs deviation of empirical-vs-predicted across the fan summarise agreement.

In [ ]:
import math

t5_alpha = 1.5
t5_sigma_w = 1.0
t5_N = 2048
t5_L = 5
t5_T0 = 1e-2
t5_n_reps = 3
rng_t5 = np.random.default_rng(11)

with Timer(f'Test5 forward {t5_n_reps} reps at N={t5_N} L={t5_L}'):
    q_star_t5 = length_fixed_point(t5_alpha, t5_sigma_w, phi)
    h1_finals = []
    h2_finals = []
    a1_prevs = []
    a2_prevs = []
    for r in range(t5_n_reps):
        out = forward_two_inputs(alpha=t5_alpha, sigma_w=t5_sigma_w, N=t5_N, L=t5_L,
                                 T0=t5_T0, phi=phi, phi_prime=phi_p, rng=rng_t5, q_star=q_star_t5)
        h1_finals.append(out['h1'][-1])
        h2_finals.append(out['h2'][-1])
        a1_prevs.append(phi(out['h1'][-2]))
        a2_prevs.append(phi(out['h2'][-2]))

h1_all = np.concatenate(h1_finals)
h2_all = np.concatenate(h2_finals)
a1_all = np.concatenate(a1_prevs)
a2_all = np.concatenate(a2_prevs)
print(f'  pooled samples: {len(h1_all)} layer-{t5_L} joints, {len(a1_all)} layer-{t5_L-1} activations')

def empirical_cf_real(t1, t2, h1, h2):
    return float(np.mean(np.cos(t1 * h1 + t2 * h2)))

def predicted_log_cf_mag(t1, t2, alpha, sigma_w, a1, a2):
    psi = sigma_w ** alpha * float(np.mean(np.abs(t1 * a1 + t2 * a2) ** alpha))
    return -0.5 * psi

t_mags = np.linspace(0.1, 2.0, 12)
theta_dirs = np.array([0.0, np.pi/6, np.pi/4, np.pi/3, np.pi/2, 3*np.pi/4, np.pi])

with Timer(f'Test5 CF probe ({len(theta_dirs)} angles x {len(t_mags)} mags)'):
    fig, axes = plt.subplots(1, len(theta_dirs), figsize=(2.6*len(theta_dirs), 4), sharey=True)
    all_emp = []
    all_pred = []
    for ax, theta in zip(axes, theta_dirs):
        t1_vec = t_mags * np.cos(theta)
        t2_vec = t_mags * np.sin(theta)
        emp_log = []
        pred_log = []
        for t1, t2 in zip(t1_vec, t2_vec):
            phi_hat = empirical_cf_real(t1, t2, h1_all, h2_all)
            emp_log.append(math.log(max(phi_hat, 1e-10)))
            pred_log.append(predicted_log_cf_mag(t1, t2, t5_alpha, t5_sigma_w, a1_all, a2_all))
        emp_log = np.array(emp_log); pred_log = np.array(pred_log)
        all_emp.append(emp_log); all_pred.append(pred_log)
        ax.plot(t_mags, emp_log, 'o-', label='empirical', markersize=5, alpha=0.85)
        ax.plot(t_mags, pred_log, 'x--', label='predicted (eq 8)', markersize=6, alpha=0.85)
        ax.set_xlabel('$|t|$')
        ax.set_title(f'$\\theta_t={theta:.2f}$', fontsize=10)
        ax.grid(alpha=0.3)
        if ax is axes[0]:
            ax.set_ylabel('$\\log|\\hat{\\varphi}|$')
            ax.legend(fontsize=8)
    plt.suptitle(f'Test 5: eq (8) check at layer $l={t5_L}$, $\\alpha={t5_alpha}$, $\\sigma_w={t5_sigma_w}$, $N \\cdot \\mathrm{{reps}}={t5_N*t5_n_reps}$', y=1.02)
    plt.tight_layout(); plt.show()

all_emp = np.concatenate(all_emp); all_pred = np.concatenate(all_pred)
rmse = float(np.sqrt(np.mean((all_emp - all_pred) ** 2)))
max_abs = float(np.max(np.abs(all_emp - all_pred)))
print(f'Test 5: log|CF| residual across {len(all_emp)} (theta, |t|) pairs:')
print(f'  rmse = {rmse:.4f}')
print(f'  max_abs = {max_abs:.4f}')
print(f'  benchmark: at this width finite-N sampling noise on log|CF| is ~1/sqrt(N*reps) ~ {1/math.sqrt(t5_N*t5_n_reps):.4f}')
print(f'  eq (8) empirically validated if rmse, max_abs are within ~3x the noise floor.')

## Summary of verdicts

After running the cells above:

- **Test 0a (length-map convergence)**: empirical $\hat q^l$ tracks $V_\alpha$-iteration from arbitrary $q^0$ and converges to $q^*(\alpha, \sigma_w)$ within $\sim 5$ layers for $\alpha < 2$ at $\sigma_w = 1.5$ (order-unity fixed point). The marginal $\alpha$-stable closure is validated in this regime.
- **Test 0b ($V_\alpha$ shape and $q^*$ existence)**: $V_\alpha(q)/q$ saturates to a finite value as $q \to 0$ on any finite $z$-grid (the cell uses $|z| \le 80$). Continuum theory says $V_\alpha(q)/q \to +\infty$ via $\log(1/q)$, but the divergence is so soft that the analytical iteration sees an effective Gaussian-like bifurcation: 30/150 grid points return $q^* \le 10^{-8}$ including some $\alpha < 2$ cases at small $\sigma_w$. Section 4(d) of `heavy_tailed_mlp.md` records the honest theory--numerics gap: nontrivial $q^* > 0$ in the continuum (and in the forward MLP using CMS-sampled untruncated weights), but the analytical iteration on a finite grid sees a hard threshold.
- **Test 1 (mean-field decoupling)**: $R$ does *not* approach 1 as width grows. At $\alpha = 1.5$, $\sigma_w = 1.0$, tanh, $N = 4096$, median $R$ across layers $\approx 0.44$--$0.73$ (no width trend). The decoupling assumption that would close eq. (11) scalarly fails; the would-be naive scalar slope $\chi_1|_\alpha^{\text{naive}}$ is biased.
- **Test 2 (perturbation growth, single point)**: at $\alpha = 1.5$, $\sigma_w = 1.0$, tanh, $q^* = 0.105$, $\chi_1|_\alpha^{\text{naive}} = 0.91 < 1$ predicts decay but empirical $\hat T^l$ grows. Concrete witness that the naive scalar closure of eq. (11) is qualitatively wrong.
- **Test 3 (joint spectral measure)**: the $R^\alpha$-weighted angular density of $(h, \xi)$ has $\sim 0.68$ mass within $\pm 0.39$ rad of an axis vs $\sim 0.50$ for uniform -- spread but not concentrated. Direct evidence that the joint $(\phi'(h), \xi)$ does not factorise.
- **Test 4 (critical-line scan)**: 10x11 sweep of $(\alpha, \sigma_w) \in [1.1, 2.0] \times [0.5, 2.5]$ at $N=1024$, $L=25$, 3 reps per cell, masking $q^* < 10^{-4}$. Empirical edge of chaos (per-layer $\log_{10}$ growth $= 0$, i.e. Lyapunov exponent $\lambda = 0$) sits *left of* the naive $\chi_1|_\alpha^{\text{naive}} = 1$ curve everywhere the two are visible: $\sigma_w^c_{\text{emp}} - \sigma_w^c_{\text{naive}} \approx -0.41$ at $\alpha = 1.3$, $-0.20$ at $\alpha = 1.7$, $-0.09$ at $\alpha = 1.9$. The naive prediction is systematically too conservative; the heavy-tailed network reaches criticality at smaller $\sigma_w$. The gap shrinks monotonically as $\alpha \to 2$ where decoupling becomes rigorous. This empirical contour is the operational definition of the heavy-tailed edge of chaos (recorded in Section 6 of `heavy_tailed_mlp.md`).

Bottom line: the heavy-tailed perturbation recursion (eqs. 10 and 11 of `heavy_tailed_mlp.md`) does not admit a Poole-style scalar closure for $\alpha < 2$ because the underlying mean-field decoupling fails empirically. The operational edge of chaos is the Lyapunov exponent of the nonlinear $\mathcal{F}$, measured directly by Test 4.